<h2>CS 3780/5780 Creative Project: </h2>
<h3>Emotion Classification of Natural Language</h3>

Names and NetIDs for your group members:

-   Dihong Luo (dl2247)
-   Tristan Feng (rf468)
-   Zhihan Chen (zc523)


<h3>Introduction:</h3>

<p> The creative project is about conducting a real-world machine learning project on your own, with everything that is involved. Unlike in the programming projects 1-5, where we gave you all the scaffolding and you just filled in the blanks, you now start from scratch. The past programming projects provide templates for how to do this (and you can reuse part of your code if you wish), and the lectures provide some of the methods you can use. So, this creative project brings realism to how you will use machine learning in the real world.  </p>

The task you will work on is classifying texts to human emotions. Through words, humans express feelings, articulate thoughts, and communicate our deepest needs and desires. Language helps us interpret the nuances of joy, sadness, anger, and love, allowing us to connect with others on a deeper level. Are you able to train an ML model that recognizes the human emotions expressed in a piece of text? <b>Please read the project description PDF file carefully and follow the instructions there. Also make sure you write your code and answers to all the questions in this Jupyter Notebook </b> </p>

<p>


<h2>Part 0: Basics</h2><p>


<h3>0.1 Import:</h3><p>
Please import necessary packages to use. Note that learning and using packages are recommended but not required for this project. Some official tutorial for suggested packacges includes:
    
https://scikit-learn.org/stable/tutorial/basic/tutorial.html
    
https://pytorch.org/tutorials/
    
https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html
<p>


In [ ]:
import re
from typing import Literal

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn import model_selection, naive_bayes, svm
from sklearn.feature_extraction import text as sklearn_text

<h3>0.2 Accuracy and Mean Squared Error:</h3><p>
To measure your performance in the Kaggle Competition, we are using accuracy. As a recap, accuracy is the percent of labels you predict correctly. To measure this, you can use library functions from sklearn. A simple example is shown below. 
<p>


In [ ]:
from sklearn.metrics import accuracy_score

y_pred = [3, 2, 1, 0, 1, 2, 3]
y_true = [0, 1, 2, 3, 1, 2, 3]
accuracy_score(y_true, y_pred)

0.42857142857142855

<h2>Part 1: Basic</h2><p>
Note that your code should be commented well and in part 1.4 you can refer to your comments.


<h3>1.1 Load and preprocess the dataset:</h3><p>
We provide how to load the data on Kaggle's Notebook.
<p>


In [ ]:
train = pd.read_csv("/kaggle/input/cs-3780-5780-how-do-you-feel/train.csv")
train_text = train["text"]
train_label = train["label"]

test = pd.read_csv("/kaggle/input/cs-3780-5780-how-do-you-feel/train.csv")
test_id = test["id"]
test_text = test["text"]

In [ ]:
train

,text,label
0,i interact with on a daily basis either in rea...,1
1,Stranger than fiction. Can't even begin to com...,1
2,i sit here with the aftermath feeling so damn ...,1
3,Great job! Hats off to you.,25
4,i hate you threads posted by people just whini...,9
...,...,...
9995,im feeling so shy,4
9996,Honestly if they were so worried about the tub...,20
9997,Don't wear out our [NAME]. We need him if this...,10
9998,Happy new year!,19


In [ ]:
test

,id,text
0,0,im feeling like a hot potato right now
1,1,i feel that are becoming impressed upon my lit...
2,2,id ever held any girls hand but boy did i sure...
3,3,i feel thats when i feel my grief over the bra...
4,4,i feel will never been resolved in a way to ke...
...,...,...
14995,14995,i feel greedy in that im looking forward to th...
14996,14996,i was feeling cold
14997,14997,Yeah deffo seen mom do that trick with a Hotdo...
14998,14998,i go to bed feeling defeated


In [ ]:
def preprocess(text: str) -> str:
    # Convert the text to lowercase
    text = text.lower()

    # Remove all non-alphabetical characters except spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
def build_vectorizer(
    train_text: np.ndarray,
    method: Literal['bow', 'tfidf'],
    min_df: float = 1,
    max_df: float = 1,
    max_features: int | None = None,
) -> sklearn_text.CountVectorizer | sklearn_text.TfidfVectorizer:
    if method == 'bow':  # Bag-of-Words vectorizer
        vectorizer = sklearn_text.CountVectorizer(
            lowercase=True,
            stop_words='english',
            max_features=max_features,
        )
    elif method == 'tfidf':  # TF-IDF vectorizer
        vectorizer = sklearn_text.TfidfVectorizer(
            lowercase=True,
            stop_words='english',
            min_df=min_df,
            max_df=max_df,
            max_features=max_features,
        )
    else:
        raise ValueError(f'Unsupported method: {method}')

    # Fit the vectorizer to the training text data
    vectorizer.fit(train_text)

    return vectorizer

<h3>1.2 Use At Least Two Training Algorithms from class:</h3><p>
You need to use at least two training algorithms from class. You can use your code from previous projects or any packages you imported in part 0.1.


In [ ]:
# Linear SVM
def train_svm(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray | None = None,
    C: float = 1,
    max_iter: int = 1000,
    random_state: int | None = 42,
) -> tuple[np.ndarray, float | None]:
    # Initialize the Linear SVM model with specified parameters
    svm_model = svm.LinearSVC(C=C, max_iter=max_iter, random_state=random_state)

    # Train the model on the training data
    svm_model.fit(X_train, y_train)

    # Predict labels for the test data
    y_pred = svm_model.predict(X_test)

    # Compute accuracy if test labels are provided
    if y_test is not None:
        accuracy = float(accuracy_score(y_test, y_pred))
    else:
        accuracy = None

    return y_pred, accuracy


# Naive Bayes
def train_nb(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray | None = None,
    alpha: float = 1,
) -> tuple[np.ndarray, float | None]:
    # Initialize the Naive Bayes model with the specified smoothing parameter
    nb_model = naive_bayes.MultinomialNB(alpha=alpha)

    # Train the Naive Bayes model on the training data
    nb_model.fit(X_train, y_train)

    # Predict labels for the test data
    y_pred = nb_model.predict(X_test)

    # Compute accuracy if test labels are provided
    if y_test is not None:
        accuracy = float(accuracy_score(y_test, y_pred))
    else:
        accuracy = None

    return y_pred, accuracy

<h3>1.3 Training, Validation and Model Selection:</h3><p>
You need to split your data to a training set and validation set or performing a cross-validation for model selection.


In [ ]:
def kfold_validation(
    train_text: np.ndarray,
    train_label: np.ndarray,
    params: dict,
    method: Literal['bow', 'tfidf'],
    model: Literal['svm', 'nb'],
    k: int = 5,
    min_df: float = 1,
    max_df: float = 1,
    max_features: int | None = None,
) -> float:
    # Initialize k-fold cross-validation splitter with k splits
    kfold = model_selection.KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in kfold.split(train_text, train_label):
        # Split the data into training and validation sets for the current fold
        X_train, X_val = train_text[train_idx], train_text[val_idx]
        y_train, y_val = train_label[train_idx], train_label[val_idx]

        # Build a vectorizer (Bag-of-Words or TF-IDF) and fit it to the training data
        vectorizer = build_vectorizer(X_train, method, min_df, max_df, max_features)

        # Transform training and validation data into feature vectors
        X_train = vectorizer.transform(X_train).toarray()  # pyright: ignore
        X_val = vectorizer.transform(X_val).toarray()  # pyright: ignore

        # Train the specified model and evaluate it on the validation set
        if model == 'svm':
            _, val_accuracy = train_svm(
                X_train,
                X_val,
                y_train,
                y_val,
                C=params['C'],
                max_iter=params['max_iter'],
            )
        elif model == 'nb':
            _, val_accuracy = train_nb(
                X_train,
                X_val,
                y_train,
                y_val,
                alpha=params['alpha'],
            )
        else:
            # Raise an error if an unsupported model type is specified
            raise ValueError(f'Unsupported model type: {model}')

        # Append the validation accuracy for the current fold to the scores list
        scores.append(val_accuracy)

    return float(np.mean(scores))

In [ ]:
def grid_search(
    train_text: np.ndarray,
    train_label: np.ndarray,
    param_grid: dict,
    method: Literal['bow', 'tfidf'],
    model: Literal['svm', 'nb'],
    k: int = 5,
    min_df: float = 1,
    max_df: float = 1,
    max_features: int | None = None,
) -> dict:
    best_score = -np.inf
    best_params = None

    # Iterate through all parameter combinations in the parameter grid
    for param_combination in model_selection.ParameterGrid(param_grid):
        val_score = kfold_validation(
            train_text,
            train_label,
            param_combination,
            method,
            model,
            k,
            min_df,
            max_df,
            max_features,
        )

        # Log the current parameter combination and its validation accuracy
        print(f'Params: {param_combination}, Validation Accuracy: {val_score}')

        # Update the best score and parameters if the current combination performs better
        if val_score > best_score:
            best_score = val_score
            best_params = param_combination

    return {'best_params': best_params, 'best_score': best_score}

In [ ]:
# Apply preprocessing to clean the training text data,
# and convert training data to a NumPy array
train_text_cleaned = np.array(train_text.apply(preprocess))
train_label = np.array(train_label)

In [ ]:
# Define the parameter grid for SVM
param_grid_svm = {'C': [0.001, 0.01, 0.1, 1], 'max_iter': [500, 1000]}

# Perform grid search to find the best hyperparameters for SVM
best_svm_result = grid_search(
    train_text_cleaned,
    train_label,
    param_grid=param_grid_svm,
    method='bow',  # Use Bag-of-Words as the feature extraction method
    model='svm',
    k=5,  # Perform 5-fold cross-validation
)

# Print the best parameters and validation accuracy found during grid search
print(f"Best SVM Parameters: {best_svm_result['best_params']}")
print(f"Best SVM Validation Accuracy: {best_svm_result['best_score']}")

Params: {'C': 0.001, 'max_iter': 500}, Validation Accuracy: 0.4338
Params: {'C': 0.001, 'max_iter': 1000}, Validation Accuracy: 0.4338
Params: {'C': 0.01, 'max_iter': 500}, Validation Accuracy: 0.6555
Params: {'C': 0.01, 'max_iter': 1000}, Validation Accuracy: 0.6555
Params: {'C': 0.1, 'max_iter': 500}, Validation Accuracy: 0.7258000000000001
Params: {'C': 0.1, 'max_iter': 1000}, Validation Accuracy: 0.7258000000000001
Params: {'C': 1, 'max_iter': 500}, Validation Accuracy: 0.7043
Params: {'C': 1, 'max_iter': 1000}, Validation Accuracy: 0.7043
Best SVM Parameters: {'C': 0.1, 'max_iter': 500}
Best SVM Validation Accuracy: 0.7258000000000001


In [ ]:
# Define the parameter grid for Naive Bayes
param_grid_nb = {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]}

# Perform grid search to find the best hyperparameters for Naive Bayes
best_nb_result = grid_search(
    train_text_cleaned,
    train_label,
    param_grid=param_grid_nb,
    method='bow',  # Use Bag-of-Words as the feature extraction method
    model='nb',
    k=5,  # Perform 5-fold cross-validation
)

# Print the best parameters and validation accuracy found during grid search
print(f"Best Naive Bayes Parameters: {best_nb_result['best_params']}")
print(f"Best Naive Bayes Accuracy: {best_nb_result['best_score']}")

Params: {'alpha': 0.01}, Validation Accuracy: 0.5406
Params: {'alpha': 0.1}, Validation Accuracy: 0.5841999999999999
Params: {'alpha': 0.5}, Validation Accuracy: 0.5776999999999999
Params: {'alpha': 1.0}, Validation Accuracy: 0.5394
Params: {'alpha': 2.0}, Validation Accuracy: 0.4936999999999999
Best Naive Bayes Parameters: {'alpha': 0.1}
Best Naive Bayes Accuracy: 0.5841999999999999


In [ ]:
# Preprocess the test text data
test_text_cleaned = np.array(test_text.apply(preprocess))

# Build a vectorizer using the training text data with Bag-of-Words (BoW) method
vectorizer = build_vectorizer(train_text_cleaned, 'bow')

# Transform the text data into numerical feature matrices
X_train = vectorizer.transform(train_text_cleaned).toarray()  # pyright: ignore
X_test = vectorizer.transform(test_text_cleaned).toarray()  # pyright: ignore

# Convert training labels to a NumPy array
y_train = np.array(train_label)

In [ ]:
# Train the SVM model using the best hyperparameters found during grid search
svm_prediction, _ = train_svm(
    X_train,
    X_test,
    y_train,
    C=best_svm_result['best_params']['C'],
    max_iter=best_svm_result['best_params']['max_iter'],
)

id = range(15000)
submission = pd.DataFrame({'id': id, 'label': svm_prediction})

# Save the submission file in the specified directory
submission.to_csv('/kaggle/working/submission_svm.csv', index=False)

In [ ]:
# Train the Naive Bayes model using the best hyperparameters found during grid search
nb_prediction, _ = train_nb(
    X_train,
    X_test,
    y_train,
    alpha=best_nb_result['best_params']['alpha'],
)

id = range(15000)
submission = pd.DataFrame({'id': id, 'label': nb_prediction})

# Save the submission file in the specified directory
submission.to_csv('/kaggle/working/submission_nb.csv', index=False)

<h3>1.4 Explanation in Words:</h3><p>
    You need to answer the following questions in the markdown cell after this cell:


1.4.1 How did you formulate the learning problem?

1.4.2 Which two learning methods from class did you choose and why did you made the choices?

1.4.3 How did you do the model selection?

1.4.4 Does the test performance reach the first baseline "Tiny Piney"? (Please include a screenshot of Kaggle Submission)


##### 1.4.1 How did you formulate the learning problem?

-   The objective is to map input text data to one of 28 predefined emotion classes. The raw text data was preprocessed to remove noise, including converting text to lowercase, removing non-alphabetical characters, and stripping extra spaces. Following this, we choose two types of data processing techniques: the Bag-of-Words (BoW) and TF-IDF vectorizer to transform the textual data into a numerical feature representation. After numerous tasts, we choose BoW for its suitabiity for different models. Then the learning task was designed as a supervised learning problem, where the model learns patterns from labeled training data (train.csv) and predicts labels for the test data (test.csv). The models were trained to minimize classification errors by optimizing their parameters on the training data. At the same time, the key objectives in formulating the learning problem are: Accurate Prediction, Computational Efficiency Interpretability and Generalization.

##### 1.4.2 Which two learning methods from class did you choose and why did you made the choices?

-   We chose Linear SVM and Naive Bayes from class. Firstly, we think Linear SVM is well-suited for high-dimensional text data like Bag-of-Words due to its efficiency and ability to create robust decision boundaries. It's also computationally manageable and scalable for the given dataset size.
    Plus,regularization through the C parameter allows us to balance complexity and performance and avoid overfitting. As for Naive Bayes, it assumes feature independence, which aligns well with Bag-of-Words representation. It can provide good baseline performance in text classification tasks.

-   We have also tried boosting method such as XGBoost and logic regression during experimentation but found to be too computationally expensive given the resource constraints on Kaggle.

##### 1.4.3 How did you do the model selection?

-   K-Fold Cross-Validation: The dataset was split into training and validation sets using 5-fold cross-validation, which ensured robust evaluation by testing the model's performance on multiple subsets of the data and reducing variance in the validation scores.

-   Grid Search: Hyperparameter tuning was performed by systematically testing combinations of parameters. For SVM, the parameters C and max_iter were tuned. For Naive Bayes, the smoothing parameter alpha was optimized. The grid search iterated over these combinations to identify the best-performing configuration based on cross-validation accuracy.

-   Prediction: The Grid Search returns the parameter which produces the best efficiency. We use this best parameter to train the model and use the trained model to predict the results.

##### 1.4.4 Does the test performance reach the first baseline "Tiny Piney"? (Please include a screenshot of Kaggle Submission)

-   Yes, it does.

<div style="text-align: center">
    <img
        alt="SVM Submission"
        src="https://github.com/Andy-Dihong-Luo/My-Image-Host/blob/master/Final%20Project/notebook_template/svm_submission_1733710593703.png?raw=true"
        width="90%"
    />
</div>


<h2>Part 2: Be creative!</h2><p>


<h3>2.1 Open-ended Code:</h3><p>
You may follow the steps in part 1 again but making innovative changes like using new training algorithms, etc. Make sure you explain everything clearly in part 2.2. Note that beating "Zero Hero" is only a small portion of this part. Any creative ideas will receive most points as long as they are reasonable and clearly explained.


In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None) -> None:
        self.encodings = encodings
        self.labels = labels

    def __len__(self) -> int:
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx) -> dict[str, torch.Tensor]:
        # Create a dictionary with tokenized inputs for the given index
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])

        return item

In [ ]:
def compute_accuracy(eval_pred) -> dict[str, float]:
    # Unpack predictions (logits) and true labels from eval_pred
    logits, labels = eval_pred.predictions, eval_pred.label_ids

    # Convert logits to predicted class indices by taking the argmax along the last dimension
    predictions = logits.argmax(axis=-1)

    # Calculate accuracy by comparing predicted classes with true labels
    accuracy = accuracy_score(labels, predictions)

    return {'accuracy': float(accuracy)}

In [ ]:
# Split the dataset into training and validation sets
train_texts, val_texts, train_labels, val_labels = model_selection.train_test_split(
    train['text'], train['label'], test_size=0.2, shuffle=True, random_state=42
)

# Load the tokenizer for BERT
tokenizer = transformers.BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the datasets with truncation and padding
train_encodings = tokenizer(
    list(train_texts), truncation=True, padding=True, max_length=128
)
val_encodings = tokenizer(
    list(val_texts), truncation=True, padding=True, max_length=128
)
test_encodings = tokenizer(
    list(test['text']), truncation=True, padding=True, max_length=128
)

# Wrap tokenized data in PyTorch datasets
train_dataset = SentimentDataset(train_encodings, list(train_labels))
val_dataset = SentimentDataset(val_encodings, list(val_labels))
test_dataset = SentimentDataset(test_encodings)

In [ ]:
# Load the pre-trained BERT model for sequence classification
model = transformers.BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=train['label'].nunique()
)

# Define training arguments
training_args = transformers.TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize the Trainer
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_accuracy,
)

# Train the model
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\miniforge3\envs\ml\Lib\site-packages\accelerate\accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': 2.9976, 'grad_norm': 6.324663162231445, 'learning_rate': 4.96e-05, 'epoch': 0.04}
{'loss': 2.4472, 'grad_norm': 4.7166852951049805, 'learning_rate': 4.92e-05, 'epoch': 0.08}
{'loss': 2.2543, 'grad_norm': 4.426314353942871, 'learning_rate': 4.88e-05, 'epoch': 0.12}
{'loss': 2.0347, 'grad_norm': 5.824423789978027, 'learning_rate': 4.8400000000000004e-05, 'epoch': 0.16}
{'loss': 1.8751, 'grad_norm': 5.4824442863464355, 'learning_rate': 4.8e-05, 'epoch': 0.2}
{'loss': 1.717, 'grad_norm': 4.250117778778076, 'learning_rate': 4.76e-05, 'epoch': 0.24}
{'loss': 1.6962, 'grad_norm': 8.712447166442871, 'learning_rate': 4.72e-05, 'epoch': 0.28}
{'loss': 1.686, 'grad_norm': 5.414229869842529, 'learning_rate': 4.6800000000000006e-05, 'epoch': 0.32}
{'loss': 1.4258, 'grad_norm': 4.522858619689941, 'learning_rate': 4.64e-05, 'epoch': 0.36}
{'loss': 1.3507, 'grad_norm': 5.739327907562256, 'learning_rate': 4.600000000000001e-05, 'epoch': 0.4}
{'loss': 1.3273, 'grad_norm': 4.929150581359863, 'le

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.9906200766563416, 'eval_accuracy': 0.741, 'eval_runtime': 3.2408, 'eval_samples_per_second': 617.123, 'eval_steps_per_second': 19.439, 'epoch': 1.0}
{'loss': 0.8553, 'grad_norm': 9.292866706848145, 'learning_rate': 3.964e-05, 'epoch': 1.04}
{'loss': 0.8295, 'grad_norm': 3.2123591899871826, 'learning_rate': 3.9240000000000004e-05, 'epoch': 1.08}
{'loss': 0.848, 'grad_norm': 5.4609503746032715, 'learning_rate': 3.884e-05, 'epoch': 1.12}
{'loss': 0.9649, 'grad_norm': 5.223743438720703, 'learning_rate': 3.8440000000000005e-05, 'epoch': 1.16}
{'loss': 0.8284, 'grad_norm': 4.045118808746338, 'learning_rate': 3.804e-05, 'epoch': 1.2}
{'loss': 0.8262, 'grad_norm': 5.452331066131592, 'learning_rate': 3.7640000000000006e-05, 'epoch': 1.24}
{'loss': 0.71, 'grad_norm': 3.912476062774658, 'learning_rate': 3.724e-05, 'epoch': 1.28}
{'loss': 0.7901, 'grad_norm': 11.989408493041992, 'learning_rate': 3.684e-05, 'epoch': 1.32}
{'loss': 0.8396, 'grad_norm': 5.7670745849609375, 'learning_r

...

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.8606656789779663, 'eval_accuracy': 0.7595, 'eval_runtime': 3.2486, 'eval_samples_per_second': 615.641, 'eval_steps_per_second': 19.393, 'epoch': 4.0}
{'loss': 0.4234, 'grad_norm': 3.2657532691955566, 'learning_rate': 9.68e-06, 'epoch': 4.04}
{'loss': 0.2932, 'grad_norm': 6.873795032501221, 'learning_rate': 9.28e-06, 'epoch': 4.08}
{'loss': 0.3823, 'grad_norm': 4.176461696624756, 'learning_rate': 8.880000000000001e-06, 'epoch': 4.12}
{'loss': 0.3613, 'grad_norm': 2.730264186859131, 'learning_rate': 8.48e-06, 'epoch': 4.16}
{'loss': 0.3263, 'grad_norm': 2.8528010845184326, 'learning_rate': 8.08e-06, 'epoch': 4.2}
{'loss': 0.3075, 'grad_norm': 1.8667662143707275, 'learning_rate': 7.68e-06, 'epoch': 4.24}
{'loss': 0.3263, 'grad_norm': 3.1212503910064697, 'learning_rate': 7.280000000000001e-06, 'epoch': 4.28}
{'loss': 0.4092, 'grad_norm': 5.206876277923584, 'learning_rate': 6.88e-06, 'epoch': 4.32}
{'loss': 0.2985, 'grad_norm': 4.010383605957031, 'learning_rate': 6.48e-06, '

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.877459704875946, 'eval_accuracy': 0.7655, 'eval_runtime': 3.2466, 'eval_samples_per_second': 616.037, 'eval_steps_per_second': 19.405, 'epoch': 5.0}
{'train_runtime': 301.7256, 'train_samples_per_second': 132.571, 'train_steps_per_second': 4.143, 'train_loss': 0.7228744524002075, 'epoch': 5.0}


TrainOutput(global_step=1250, training_loss=0.7228744524002075, metrics={'train_runtime': 301.7256, 'train_samples_per_second': 132.571, 'train_steps_per_second': 4.143, 'total_flos': 1418664133440000.0, 'train_loss': 0.7228744524002075, 'epoch': 5.0})

In [ ]:
# Use the trained model to make predictions on the test dataset
predictions = trainer.predict(test_dataset)

# Extract the predicted class labels from the model's output logits
predicted_labels = torch.argmax(
    torch.tensor(predictions.predictions), axis=1  # pyright: ignore
)

# Add the predicted labels as a new column in the test DataFrame
test['label'] = predicted_labels.numpy()

# Save the results (id and label) to a CSV file for submission
test[['id', 'label']].to_csv('submission_bert.csv', index=False)

  0%|          | 0/469 [00:00<?, ?it/s]

In [ ]:
# Split the dataset into training and validation sets
train_texts, val_texts, train_labels, val_labels = model_selection.train_test_split(
    train['text'], train['label'], test_size=0.2, shuffle=True, random_state=42
)

# Load the tokenizer for DeBERTaV3
tokenizer = transformers.AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

# Tokenize the datasets with truncation and padding
train_encodings = tokenizer(
    list(train_texts), truncation=True, padding=True, max_length=128
)
val_encodings = tokenizer(
    list(val_texts), truncation=True, padding=True, max_length=128
)
test_encodings = tokenizer(
    list(test['text']), truncation=True, padding=True, max_length=128
)

# Wrap tokenized data in PyTorch datasets
train_dataset = SentimentDataset(train_encodings, list(train_labels))
val_dataset = SentimentDataset(val_encodings, list(val_labels))
test_dataset = SentimentDataset(test_encodings)

In [ ]:
# Load the pre-trained DeBERTaV3 model for sequence classification
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base', num_labels=train['label'].nunique()
)

# Define training arguments
training_args = transformers.TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
)

# Initialize the Trainer
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_accuracy,
)

# Train the model
trainer.train()

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
d:\miniforge3\envs\ml\Lib\site-packages\accelerate\accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': 3.1563, 'grad_norm': 4.6700758934021, 'learning_rate': 4.96e-05, 'epoch': 0.04}
{'loss': 2.543, 'grad_norm': 4.568312644958496, 'learning_rate': 4.92e-05, 'epoch': 0.08}
{'loss': 2.3188, 'grad_norm': 4.94827127456665, 'learning_rate': 4.88e-05, 'epoch': 0.12}
{'loss': 2.128, 'grad_norm': 6.527798652648926, 'learning_rate': 4.8400000000000004e-05, 'epoch': 0.16}
{'loss': 1.9475, 'grad_norm': 9.281331062316895, 'learning_rate': 4.8e-05, 'epoch': 0.2}
{'loss': 1.8138, 'grad_norm': 4.225669860839844, 'learning_rate': 4.76e-05, 'epoch': 0.24}
{'loss': 1.736, 'grad_norm': 11.278286933898926, 'learning_rate': 4.72e-05, 'epoch': 0.28}
{'loss': 1.7948, 'grad_norm': 8.859150886535645, 'learning_rate': 4.6800000000000006e-05, 'epoch': 0.32}
{'loss': 1.4834, 'grad_norm': 5.010193347930908, 'learning_rate': 4.64e-05, 'epoch': 0.36}
{'loss': 1.4954, 'grad_norm': 7.206892490386963, 'learning_rate': 4.600000000000001e-05, 'epoch': 0.4}
{'loss': 1.5395, 'grad_norm': 4.918926239013672, 'learnin

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 1.3096539974212646, 'eval_accuracy': 0.6055, 'eval_runtime': 4.6813, 'eval_samples_per_second': 427.229, 'eval_steps_per_second': 13.458, 'epoch': 1.0}
{'loss': 1.1955, 'grad_norm': 17.039634704589844, 'learning_rate': 3.964e-05, 'epoch': 1.04}
{'loss': 1.1416, 'grad_norm': 4.440891265869141, 'learning_rate': 3.9240000000000004e-05, 'epoch': 1.08}
{'loss': 1.0459, 'grad_norm': 6.883886814117432, 'learning_rate': 3.884e-05, 'epoch': 1.12}
{'loss': 1.1967, 'grad_norm': 9.190842628479004, 'learning_rate': 3.8440000000000005e-05, 'epoch': 1.16}
{'loss': 1.0243, 'grad_norm': 3.876006841659546, 'learning_rate': 3.804e-05, 'epoch': 1.2}
{'loss': 1.0118, 'grad_norm': 9.040027618408203, 'learning_rate': 3.7640000000000006e-05, 'epoch': 1.24}
{'loss': 0.9214, 'grad_norm': 4.258076190948486, 'learning_rate': 3.724e-05, 'epoch': 1.28}
{'loss': 0.9585, 'grad_norm': 3.3920304775238037, 'learning_rate': 3.684e-05, 'epoch': 1.32}
{'loss': 1.0575, 'grad_norm': 6.710408687591553, 'learning

...

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.8551611304283142, 'eval_accuracy': 0.7635, 'eval_runtime': 4.6477, 'eval_samples_per_second': 430.322, 'eval_steps_per_second': 13.555, 'epoch': 4.0}
{'loss': 0.6937, 'grad_norm': 8.048962593078613, 'learning_rate': 9.68e-06, 'epoch': 4.04}
{'loss': 0.5431, 'grad_norm': 5.96552038192749, 'learning_rate': 9.28e-06, 'epoch': 4.08}
{'loss': 0.5305, 'grad_norm': 3.7225241661071777, 'learning_rate': 8.880000000000001e-06, 'epoch': 4.12}
{'loss': 0.6127, 'grad_norm': 3.8012313842773438, 'learning_rate': 8.48e-06, 'epoch': 4.16}
{'loss': 0.6017, 'grad_norm': 3.071937322616577, 'learning_rate': 8.08e-06, 'epoch': 4.2}
{'loss': 0.5676, 'grad_norm': 4.908409118652344, 'learning_rate': 7.68e-06, 'epoch': 4.24}
{'loss': 0.5614, 'grad_norm': 2.694415807723999, 'learning_rate': 7.280000000000001e-06, 'epoch': 4.28}
{'loss': 0.6363, 'grad_norm': 4.668781280517578, 'learning_rate': 6.88e-06, 'epoch': 4.32}
{'loss': 0.512, 'grad_norm': 3.0294878482818604, 'learning_rate': 6.48e-06, 'epo

  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 0.8544384241104126, 'eval_accuracy': 0.762, 'eval_runtime': 4.6585, 'eval_samples_per_second': 429.32, 'eval_steps_per_second': 13.524, 'epoch': 5.0}
{'train_runtime': 2327.5895, 'train_samples_per_second': 17.185, 'train_steps_per_second': 0.537, 'train_loss': 0.9326839370727539, 'epoch': 5.0}


TrainOutput(global_step=1250, training_loss=0.9326839370727539, metrics={'train_runtime': 2327.5895, 'train_samples_per_second': 17.185, 'train_steps_per_second': 0.537, 'total_flos': 1418689569600000.0, 'train_loss': 0.9326839370727539, 'epoch': 5.0})

In [ ]:
# Use the trained model to make predictions on the test dataset
predictions = trainer.predict(test_dataset)

# Extract the predicted class labels from the model's output logits
predicted_labels = torch.argmax(
    torch.tensor(predictions.predictions), axis=1  # pyright: ignore
)

# Add the predicted labels as a new column in the test DataFrame
test['label'] = predicted_labels.numpy()

# Save the results (id and label) to a CSV file for submission
test[['id', 'label']].to_csv('submission_deberta-v3.csv', index=False)

  0%|          | 0/469 [00:00<?, ?it/s]

<h3>2.2 Explanation in Words:</h3><p>
You need to answer the following questions in a markdown cell after this cell:


2.2.1 How much did you manage to improve performance on the test set? Did you beat "Zero Hero" in Kaggle? (Please include a screenshot of Kaggle Submission)

2.2.2 Please explain in detail how you achieved this and what you did specifically and why you tried this.


##### 2.2.1 How much did you manage to improve performance on the test set? Did you beat "Zero Hero" in Kaggle?

-   Yes, we managed to beat "Zero Hero" on Kaggle. Initially, using Bag-of-Words and SVM, we achieved an accuracy of 72.15%. After transitioning to BERT, the accuracy significantly improved to 76.36%. Finally, by using DeBERTa-v3, we achieved a slight improvement, reaching 77.19%. We used BERT because it processes the data bidirectionally, leading to a more nuanced understanding to word relationships.

<div style="text-align: center">
    <img
        alt="DeBERTaV3 Submission"
        src="https://github.com/Andy-Dihong-Luo/My-Image-Host/blob/master/Final%20Project/notebook_template/debertav3_submission_1733710363610.png?raw=true"
        width="90%"
    />
</div>

##### 2.2.2 Please explain in detail how you achieved this and what you did specifically and why you tried this.

1. Initial Challenges with Bag-of-Words and old models:

    - We started with a Bag-of-Words approach and SVM, achieving a test accuracy of 72.15%. However, the large number of features generated caused several challenges, including increased computational complexity, memory usage, and sparsity issues. We attempted to limit the number of features to improve SVM's performance, but observed no significant gains. This is likely because reducing the feature set removed important information, thereby limiting the model's ability to generalize effectively. We also tried to use Multidimensional Naive Bayes, but the training accuracy is low, about 60%. We think a possible reason is that Naive Bayes algorithm assumes the datasets are conditionally dependent. However, in reality, it may not.

2. Class Imbalance:

    - We noticed severe class imbalance in the dataset, where some labels were heavily underrepresented, causing the model to favor the majority class. We tried using `imbalanced-learn` to perform oversampling, but since one label had only 3 samples, oversampling (especially k-nearest neighbors-based methods) performed poorly.

3. Transition to BERT:

    - We transitioned to a pre-trained BERT model to address the limitations of Bag-of-Words and TF-IDF. Both Bag-of-Words and TF-IDF representations tend to produce sparse feature matrices, which can hinder model performance and generalization. BERT, on the other hand, uses a subword tokenization method and learns dense contextual embeddings, making it better suited for sentence-level sentiment analysis. Its ability to capture the meaning of words in context ensures that subtle relationships, such as negations or sentiment nuances, are effectively modeled. This transition significantly improved test accuracy to 76.36%.
    - We experimented with increasing the number of epochs but found that while the training loss decreased, validation accuracy plateaued, indicating a risk of overfitting. We decided to use 5 epochs as the optimal choice.

4. Using DeBERTa-v3:
    - Finally, we switched to DeBERTa-v3, a more advanced transformer model designed to better capture linguistic nuances and token relationships. DeBERTa improves upon BERT by using disentangled attention mechanisms, where word content and word position are encoded separately. This allows DeBERTa to capture sentence structure and context more effectively, which is crucial for nuanced tasks like sentiment analysis.
    - DeBERTa-v3 also leverages a more efficient pre-training objective, which further enhances its ability to generalize on downstream tasks. By transitioning to this model, we achieved a small but meaningful improvement, increasing test accuracy to 77.19%.
    - We opted for the `base` variant of DeBERTa-v3 rather than the larger model (`deberta-v3-large`) to balance performance and computational cost. The `large` model requires significantly more resources for both training and inference, which we deemed unnecessary given the relatively small performance gain it might bring in our use case.


<h2>Part 3: Kaggle Submission</h2><p>
You need to generate a prediction CSV using the following cell from your trained model and submit the direct output of your code to Kaggle. The results should be presented in two columns in csv format: the first column is the data id (0-14999) and the second column includes the predictions for the test set. The first column must be named id and the second column must be named label (otherwise your submission will fail). A sample predication file can be downloaded from Kaggle for each problem. 
We provide how to save a csv file if you are running Notebook on Kaggle.


In [ ]:
# id = range(15000)
# prediction = range(15000)
# submission = pd.DataFrame({'id': id, 'label': prediction})
# submission.to_csv('/kaggle/working/submission.csv', index=False)

In [ ]:
# You may use pandas to generate a dataframe with country, date and your predictions first
# and then use to_csv to generate a CSV file.

<h2>Part 4: Resources and Literature Used</h2><p>


Please cite the papers and open resources you used.


@misc{wolf2020huggingfacestransformersstateoftheartnatural,
title={HuggingFace's Transformers: State-of-the-art Natural Language Processing},
author={Thomas Wolf and Lysandre Debut and Victor Sanh and Julien Chaumond and Clement Delangue and Anthony Moi and Pierric Cistac and Tim Rault and Rémi Louf and Morgan Funtowicz and Joe Davison and Sam Shleifer and Patrick von Platen and Clara Ma and Yacine Jernite and Julien Plu and Canwen Xu and Teven Le Scao and Sylvain Gugger and Mariama Drame and Quentin Lhoest and Alexander M. Rush},
year={2020},
eprint={1910.03771},
archivePrefix={arXiv},
primaryClass={cs.CL},
url={https://arxiv.org/abs/1910.03771},
}

@article{DBLP:journals/corr/abs-1810-04805,
author = {Jacob Devlin and
Ming{-}Wei Chang and
Kenton Lee and
Kristina Toutanova},
title = {{BERT:} Pre-training of Deep Bidirectional Transformers for Language
Understanding},
journal = {CoRR},
volume = {abs/1810.04805},
year = {2018},
url = {http://arxiv.org/abs/1810.04805},
archivePrefix = {arXiv},
eprint = {1810.04805},
timestamp = {Tue, 30 Oct 2018 20:39:56 +0100},
biburl = {https://dblp.org/rec/journals/corr/abs-1810-04805.bib},
bibsource = {dblp computer science bibliography, https://dblp.org}
}

@misc{he2021debertav3,
title={DeBERTaV3: Improving DeBERTa using ELECTRA-Style Pre-Training with Gradient-Disentangled Embedding Sharing},
author={Pengcheng He and Jianfeng Gao and Weizhu Chen},
year={2021},
eprint={2111.09543},
archivePrefix={arXiv},
primaryClass={cs.CL}
}

@inproceedings{
he2021deberta,
title={DEBERTA: DECODING-ENHANCED BERT WITH DISENTANGLED ATTENTION},
author={Pengcheng He and Xiaodong Liu and Jianfeng Gao and Weizhu Chen},
booktitle={International Conference on Learning Representations},
year={2021},
url={https://openreview.net/forum?id=XPZIaotutsD}
}
